# 04. Poisson Model PRO

Modelado probabilístico de goles:
 - Predicción de goles esperados (λ_home, λ_away)
 - Generación de matriz de probabilidades
 - Derivación de probabilidades 1X2

In [5]:
from collections import defaultdict, deque
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import poisson

from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, log_loss, mean_poisson_deviance
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

dataPath = Path("../data/processed/model_dataset.csv")
cleanPath = Path("../data/processed/clean_matches.csv")

df = pd.read_csv(dataPath, parse_dates=["date"])

print("Dataset base de features:", df.shape)
df.head()

Dataset base de features: (47769, 24)


,date,homeTeam,awayTeam,eloHome,eloAway,eloDiff,absEloDiff,eloExpectedHomeWin,homeMatchesPlayed,awayMatchesPlayed,homeWinRate,awayWinRate,winRateDiff,homeGoalsForAvg,awayGoalsForAvg,goalsForDiff,homeGoalsAgainstAvg,awayGoalsAgainstAvg,goalsAgainstDiff,attackDiff,defenseDiff,tournamentWeight,neutral,target
0,1877-03-03,England,Scotland,1496.949971,1506.023694,-9.073723,9.073723,0.486945,5,6,0.200000,0.500000,-0.300000,1.400000,2.166667,-0.766667,1.80,1.166667,0.633333,0.233333,-0.366667,0.3,0,awayWin
1,1878-03-02,Scotland,England,1511.842486,1494.028302,17.814184,17.814184,0.525614,8,6,0.625000,0.166667,0.458333,2.250000,1.333333,0.916667,1.00,2.000000,-1.000000,0.250000,-0.333333,0.3,0,homeWin
2,1879-04-05,England,Scotland,1494.183063,1517.511482,-23.328419,23.328419,0.466478,8,10,0.250000,0.700000,-0.450000,1.500000,3.400000,-1.900000,2.50,1.000000,1.500000,0.500000,-0.900000,0.3,0,homeWin
3,1880-03-13,Scotland,England,1517.086224,1497.384194,19.702030,19.702030,0.528323,12,9,0.666667,0.333333,0.333333,3.416667,1.888889,1.527778,1.25,2.666667,-1.416667,0.750000,-0.638889,0.3,0,homeWin
4,1880-03-15,Wales,England,1485.529582,1494.554133,-9.024551,9.024551,0.487016,5,10,0.000000,0.300000,-0.300000,0.200000,2.100000,-1.900000,4.00,2.900000,1.100000,-2.700000,1.900000,0.3,0,awayWin


## 2. Selección de features

In [3]:
# ## 1. Plantilla local de grupos

# %%

#groupsTemplate = {
#    "A": ["Mexico", "South Korea", "South Africa", "UEFA Playoff D winner"],
#    "B": ["Canada", "Switzerland", "Qatar", "UEFA Playoff A winner"],
#    "C": ["Brazil", "Morocco", "Scotland", "Haiti"],
#    "D": ["USA", "Paraguay", "Australia", "UEFA Playoff C winner"],
#    "E": ["Germany", "Ecuador", "Ivory Coast", "Curaçao"],
#    "F": ["Netherlands", "Japan", "Tunisia", "UEFA Playoff B winner"],
#    "G": ["Belgium", "Iran", "Egypt", "New Zealand"],
#    "H": ["Spain", "Uruguay", "Saudi Arabia", "Cape Verde"],
#    "I": ["France", "Senegal", "Norway", "Interconfed Playoff 2 winner"],
#    "J": ["Argentina", "Austria", "Algeria", "Jordan"],
#    "K": ["Portugal", "Colombia", "Uzbekistan", "Interconfed Playoff 1 winner"],
#    "L": ["England", "Croatia", "Panama", "Ghana"],
#}

#print("Grupos plantilla cargados:", {k: len(v) for k, v in groupsTemplate.items()})

In [4]:
# %%
features = [
    "eloDiff",
    "absEloDiff",
    "eloExpectedHomeWin",
    "homeWinRate",
    "awayWinRate",
    "winRateDiff",
    "homeGoalsForAvg",
    "awayGoalsForAvg",
    "goalsForDiff",
    "homeGoalsAgainstAvg",
    "awayGoalsAgainstAvg",
    "goalsAgainstDiff",
    "attackDiff",
    "defenseDiff",
    "tournamentWeight",
    "neutral"
]

missingFeatures = [col for col in features if col not in df.columns]
assert not missingFeatures, f"Faltan features en model_dataset.csv: {missingFeatures}"

dfModel = df.dropna(subset=features).copy()
X = dfModel[features]

print("Shape feature matrix:", X.shape)
print("Los targets homeScore/awayScore se construyen mas abajo desde clean_matches.csv para calibrar Poisson.")

## 3. Split temporal (CRÍTICO)


In [ ]:
# %%
splitDate = "2018-01-01"

trainMask = dfModel["date"] < splitDate
testMask = dfModel["date"] >= splitDate

X_train, X_test = X[trainMask], X[testMask]
yHome_train, yHome_test = yHome[trainMask], yHome[testMask]
yAway_train, yAway_test = yAway[trainMask], yAway[testMask]

print("Train:", X_train.shape)
print("Test:", X_test.shape)

## 4. Modelos Poisson (core)

In [ ]:
# %%
homeModel = PoissonRegressor(alpha=0.001, max_iter=300)
awayModel = PoissonRegressor(alpha=0.001, max_iter=300)

homeModel.fit(X_train, yHome_train)
awayModel.fit(X_train, yAway_train)

## 5. Predicción de goles esperados

In [ ]:
# %%
lambdaHome = homeModel.predict(X_test)
lambdaAway = awayModel.predict(X_test)

predDf = X_test.copy()
predDf["lambdaHome"] = lambdaHome
predDf["lambdaAway"] = lambdaAway

predDf.head()